# Exercise 5 — Group Recommender Systems
### Preference Aggregation Strategies
> **Recommender Systems · University of Fribourg · Spring 2026**  
> **Professor : PD Dr Luis Terán**   
> **Student : Allizha Theiventhiram**  

---

This notebook reproduces all results reported in the written submission.  
The pipeline applies three aggregation strategies — **Average**, **Least Misery**, and **Most Pleasure** — to two selected user groups, then predicts ratings for unseen movies via cosine similarity between genre vectors.


## Imports

In [58]:
import sys
!{sys.executable} -m pip install openpyxl
import openpyxl
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 45)
pd.set_option("display.float_format", "{:.4f}".format)


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


## 1 · Load the Dataset

The Excel file contains four sheets:
- **Movies Vector Representation** — 18-dimensional binary genre vectors for new and rated movies  
- **Users-Movie Ratings** — raw ratings + filled matrix (missing values replaced by movie average)  
- **Similarity RatedMovies & NewMov** — pre-computed cosine similarities (used for verification)  
- **Example** — a worked example for Group 2


In [59]:
wb = openpyxl.load_workbook("GRS_dataset.xlsx", data_only=True)
print("Sheets:", wb.sheetnames)


Sheets: ['Movies Vector Representation', 'Similarity RatedMovies & NewMov', 'Users-Movie Ratings', 'Example']


### 1a · Genre Vectors

In [60]:
ws_mv   = wb["Movies Vector Representation"]
mv_rows = list(ws_mv.iter_rows(values_only=True))

GENRES = [
    "Action","Adventure","Animation","Children","Comedy","Crime",
    "Documentary","Drama","Fantasy","FilmNoir","Horror","Musical",
    "Mystery","Romance","SciFi","Thriller","War","Western"
]

new_movies   = {}   # id (1-15)  -> (title, np.array)
rated_movies = {}   # id (0-49)  -> (title, np.array)
in_rated = False

for row in mv_rows:
    if row[0] == "MOVIES RATED BY USERS":
        in_rated = True
    mid, title = row[1], row[2]
    if not isinstance(mid, (int, float)) or title is None:
        continue
    title = str(title)          # "300" is stored as integer — cast to str
    vec = np.array([float(v) if isinstance(v, (int, float)) else 0.0
                    for v in row[3:21]])
    (rated_movies if in_rated else new_movies)[int(mid)] = (title, vec)

print(f"New movies  : {len(new_movies)}")
print(f"Rated movies: {len(rated_movies)}")


New movies  : 15
Rated movies: 50


In [61]:
df_new = pd.DataFrame(
    [(mid, t, *vec.astype(int).tolist())
     for mid, (t, vec) in sorted(new_movies.items())],
    columns=["ID", "Title"] + GENRES
)
print("New movies (genre vectors) — first 9 genres shown:")
df_new[["ID", "Title"] + GENRES[:9]]


New movies (genre vectors) — first 9 genres shown:


,ID,Title,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy
0,1,"Incredible Hulk, The",1,0,0,0,0,0,0,0,1
1,2,Indiana Jones and the Kingdom of the Crys...,1,1,0,0,1,0,0,0,0
2,3,Kung Fu Panda,1,0,1,0,1,0,0,0,0
3,4,Forgetting Sarah Marshall,0,0,0,0,1,0,0,0,0
4,5,Burn After Reading,0,0,0,0,1,1,0,0,0
5,6,Quantum of Solace,1,1,0,0,0,0,0,0,0
6,7,Get Smart,0,0,0,0,1,0,0,0,0
7,8,"10,000 B.C.",1,1,0,0,0,0,0,1,0
8,9,HellBoy II: The Golden Army,1,1,0,0,1,0,0,0,1
9,10,Tropic Thunder,1,1,0,0,1,0,0,0,0


In [62]:
df_rated = pd.DataFrame(
    [(mid, t, *vec.astype(int).tolist())
     for mid, (t, vec) in sorted(rated_movies.items())],
    columns=["ID", "Title"] + GENRES
)
print("Rated movies (genre vectors) — first 9 genres shown:")
df_rated[["ID", "Title"] + GENRES[:9]]


Rated movies (genre vectors) — first 9 genres shown:


,ID,Title,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy
0,0,"Dark Knight, The",1,0,0,0,0,1,0,1,0
1,1,Lord of the Rings: The Return of the King...,1,1,0,0,0,0,0,0,1
2,2,"Bourne Ultimatum, The",1,1,0,0,0,0,0,0,0
3,3,WALL·E,0,1,1,0,1,0,0,0,0
4,4,Finding Nemo,0,1,1,0,1,0,0,0,0
5,5,"Incredibles, The",1,1,1,0,1,0,0,0,0
6,6,Ratatouille,0,0,1,0,1,0,0,0,1
7,7,"Monsters, Inc.",0,0,1,0,1,0,0,0,1
8,8,Shrek,0,1,1,0,1,0,0,0,1
9,9,Schindler's List,0,0,0,0,0,0,0,1,0


### 1b · Filled Ratings Matrix

In [63]:
ws_ur   = wb["Users-Movie Ratings"]
ur_rows = list(ws_ur.iter_rows(values_only=True))

header_idx = next(i for i, r in enumerate(ur_rows)
                  if r[0] == "RATING MATRIX - NO EMPTY SPACES") + 1

user_cols = {int(str(v)[1:]): j
             for j, v in enumerate(ur_rows[header_idx])
             if v and str(v).startswith("U")}

movie_ratings = {}
for row in ur_rows[header_idx + 1:]:
    if not isinstance(row[0], (int, float)):
        continue
    mid = int(row[0])
    movie_ratings[mid] = {u: float(row[c])
                          for u, c in user_cols.items()
                          if isinstance(row[c], (int, float))}

print(f"Users in matrix : {len(user_cols)}")
print(f"Rated movies    : {len(movie_ratings)}")


Users in matrix : 58
Rated movies    : 50


In [64]:
# Slice: first 10 movies × first 10 users
users_show  = sorted(user_cols.keys())[:10]
movies_show = sorted(movie_ratings.keys())[:10]

matrix_slice = pd.DataFrame(
    {f"U{u}": [movie_ratings[m].get(u, None) for m in movies_show]
     for u in users_show},
    index=[rated_movies[m][0][:30] for m in movies_show]
)
matrix_slice.index.name = "Movie"
print("Filled ratings matrix — first 10 movies × first 10 users:")
matrix_slice


Filled ratings matrix — first 10 movies × first 10 users:


,U1,U2,U3,U4,U5,U6,U7,U8,U9,U10
Movie,,,,,,,,,,
"Dark Knight, The",4.0000,2.5000,2.5000,3.0000,3.0000,5.0000,4.5000,3.0000,4.5000,4.5000
Lord of the Rings: The Return,3.5000,3.0000,0.0000,3.5000,3.8646,4.0000,2.5000,5.0000,3.5000,4.0000
"Bourne Ultimatum, The",3.5000,3.6875,2.5000,4.5000,3.5000,3.0000,2.5000,3.5000,4.5000,3.0000
WALL·E,1.5000,4.5000,3.7903,3.7903,3.7903,2.5000,4.5000,4.5000,3.5000,4.5000
Finding Nemo,2.5000,3.5000,3.5000,3.5000,3.8365,2.0000,4.5000,5.0000,3.5000,4.5000
"Incredibles, The",2.5000,3.0000,4.5000,3.4022,3.4022,3.0000,4.0000,4.0000,3.4022,2.5000
Ratatouille,3.5000,3.5000,3.0000,3.2941,3.2941,2.0000,4.5000,4.5000,2.5000,3.0000
"Monsters, Inc.",4.0000,4.5000,5.0000,3.9043,3.9043,3.9043,4.5000,4.0000,4.0000,1.5000
Shrek,4.0000,4.5000,5.0000,4.5000,3.0000,3.5000,5.0000,4.5000,3.5000,2.0000


## 2 · Cosine Similarity

For every pair (new movie, rated movie):

$$\mathrm{sim}(A,B) = \frac{\mathbf{v}_A \cdot \mathbf{v}_B}{\|\mathbf{v}_A\|\,\|\mathbf{v}_B\|}$$


In [65]:
def cosine_sim(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(np.dot(a, b) / (na * nb)) if na > 0 and nb > 0 else 0.0

sim_table = {
    nid: {rid: cosine_sim(new_movies[nid][1], rated_movies[rid][1])
          for rid in rated_movies}
    for nid in new_movies
}

# Spot-check against the dataset's Similarity sheet
print(f"sim(Dark Knight [0], 10,000 B.C. [8]) = {sim_table[8][0]:.6f}  (expected ≈ 0.666667)")
print(f"sim(WALL-E [3],      Kung Fu Panda [3]) = {sim_table[3][3]:.6f}  (expected ≈ 0.516398)")


sim(Dark Knight [0], 10,000 B.C. [8]) = 0.666667  (expected ≈ 0.666667)
sim(WALL-E [3],      Kung Fu Panda [3]) = 0.516398  (expected ≈ 0.516398)


In [66]:
# Similarity matrix as DataFrame (new movies × rated movies)
df_sim = pd.DataFrame(
    index=[f"M{nid}: {new_movies[nid][0][:20]}" for nid in sorted(new_movies)],
    columns=[f"R{rid}: {rated_movies[rid][0][:18]}" for rid in sorted(rated_movies)],
    data=[[sim_table[nid][rid] for rid in sorted(rated_movies)]
          for nid in sorted(new_movies)]
).round(3)

print("Cosine similarity — new movies (rows) × rated movies (first 10 cols):")
df_sim.iloc[:, :10]


Cosine similarity — new movies (rows) × rated movies (first 10 cols):


,"R0: Dark Knight, The",R1: Lord of the Rings:,"R2: Bourne Ultimatum,",R3: WALL·E,R4: Finding Nemo,"R5: Incredibles, The",R6: Ratatouille,"R7: Monsters, Inc.",R8: Shrek,R9: Schindler's List
"M1: Incredible Hulk, The",0.3330,0.6670,0.2890,0.2580,0.0000,0.2890,0.3330,0.3330,0.2580,0.0000
M2: Indiana Jones and th,0.3330,0.6670,0.5770,0.5160,0.6670,0.8660,0.3330,0.3330,0.5160,0.0000
M3: Kung Fu Panda,0.3330,0.3330,0.2890,0.5160,0.6670,0.8660,0.6670,0.6670,0.5160,0.0000
M4: Forgetting Sarah Mar,0.0000,0.0000,0.0000,0.6320,0.4080,0.3540,0.4080,0.4080,0.6320,0.0000
M5: Burn After Reading,0.4080,0.0000,0.0000,0.3160,0.4080,0.3540,0.4080,0.4080,0.3160,0.0000
M6: Quantum of Solace,0.3330,0.6670,0.8660,0.2580,0.3330,0.5770,0.0000,0.0000,0.2580,0.0000
M7: Get Smart,0.0000,0.0000,0.0000,0.4470,0.5770,0.5000,0.5770,0.5770,0.4470,0.0000
"M8: 10,000 B.C.",0.6670,0.6670,0.5770,0.2580,0.3330,0.5770,0.0000,0.0000,0.2580,0.4080
M9: HellBoy II: The Gold,0.2580,0.7750,0.4470,0.6000,0.5160,0.6710,0.5160,0.5160,0.6000,0.0000
M10: Tropic Thunder,0.2890,0.5770,0.5000,0.4470,0.5770,0.7500,0.2890,0.2890,0.4470,0.3540


## 3 · Aggregation Strategies

| Strategy | Formula |
|---|---|
| **Average** | $r_G^{\text{avg}}(i) = \frac{1}{|G|}\sum_{u\in G} r_u(i)$ |
| **Least Misery** | $r_G^{\text{LM}}(i) = \min_{u\in G}\, r_u(i)$ |
| **Most Pleasure** | $r_G^{\text{MP}}(i) = \max_{u\in G}\, r_u(i)$ |


In [67]:
def aggregate(group_users, strategy):
    result = {}
    for mid, ratings in movie_ratings.items():
        vals = [ratings[u] for u in group_users if u in ratings]
        if not vals:
            continue
        if   strategy == "average":
            result[mid] = float(np.mean(vals))
        elif strategy == "least_misery":
            result[mid] = float(min(vals))
        elif strategy == "most_pleasure":
            result[mid] = float(max(vals))
    return result

# Preview for Group 3: individual ratings vs all three aggregated group ratings
GROUP3 = [11, 15, 29]
strategies = ["average", "least_misery", "most_pleasure"]

rows = []
for mid in sorted(movie_ratings.keys())[:8]:
    title = rated_movies[mid][0][:30]
    indiv = [round(movie_ratings[mid].get(u, float("nan")), 2) for u in GROUP3]
    agg   = {s: round(aggregate(GROUP3, s).get(mid, float("nan")), 4) for s in strategies}
    rows.append([mid, title] + indiv + [agg[s] for s in strategies])

df_agg = pd.DataFrame(rows,
    columns=["ID", "Movie", "U11", "U15", "U29",
             "Average", "Least Misery", "Most Pleasure"])
print("Group 3 — individual vs aggregated ratings (first 8 movies):")
df_agg


Group 3 — individual vs aggregated ratings (first 8 movies):


,ID,Movie,U11,U15,U29,Average,Least Misery,Most Pleasure
0,0,"Dark Knight, The",4.0000,3.8600,4.5000,4.1204,3.8611,4.5000
1,1,Lord of the Rings: The Return,4.5000,5.0000,2.5000,4.0000,2.5000,5.0000
2,2,"Bourne Ultimatum, The",4.0000,5.0000,4.5000,4.5000,4.0000,5.0000
3,3,WALL·E,5.0000,4.5000,4.5000,4.6667,4.5000,5.0000
4,4,Finding Nemo,5.0000,5.0000,3.0000,4.3333,3.0000,5.0000
5,5,"Incredibles, The",4.5000,4.5000,3.4000,4.1341,3.4022,4.5000
6,6,Ratatouille,4.5000,4.0000,3.0000,3.8333,3.0000,4.5000
7,7,"Monsters, Inc.",4.5000,5.0000,3.5000,4.3333,3.5000,5.0000


## 4 · Predicted Ratings for New Movies

$$\hat{r}_G(\text{new}) = \max_{\text{rated}}\!\bigl(r_G(\text{rated}) \times \mathrm{sim}(\text{new},\,\text{rated})\bigr)$$


In [68]:
def predict_all(group_users, strategy):
    gr = aggregate(group_users, strategy)
    rows = []
    for nid in sorted(new_movies):
        title = new_movies[nid][0]
        best_score, best_rated = 0.0, None
        for rid, g_rating in gr.items():
            score = g_rating * sim_table[nid][rid]
            if score > best_score:
                best_score, best_rated = score, rated_movies[rid][0]
        rows.append({
            "ID"              : nid,
            "New Movie"       : title,
            "Best Rated Movie": best_rated,
            "Predicted Score" : best_score
        })
    df = (pd.DataFrame(rows)
            .sort_values("Predicted Score", ascending=False)
            .reset_index(drop=True))
    df.index += 1
    df.index.name = "Rank"
    return df


## 5 · Results — Group 3 (U11, U15, U29)

### Strategy: Average

In [69]:
df_g3_average = predict_all([11, 15, 29], "average")
print("Group 3 | Average — full ranking:")
df_g3_average


Group 3 | Average — full ranking:


,ID,New Movie,Best Rated Movie,Predicted Score
Rank,,,,
1,9,HellBoy II: The Golden Army,Star Wars: Episode III - Revenge of the Sith,4.1740
2,1,"Incredible Hulk, The",Star Wars: Episode III - Revenge of the Sith,4.0415
3,6,Quantum of Solace,"Bourne Ultimatum, The",3.8971
4,2,Indiana Jones and the Kingdom of the Crys...,Pirates of the Caribbean: The Curse of th...,3.7528
5,4,Forgetting Sarah Marshall,There's Something About Mary,3.6667
6,11,Sex and the City,There's Something About Mary,3.6667
7,3,Kung Fu Panda,"Incredibles, The",3.5802
8,5,Burn After Reading,Pulp Fiction,3.3947
9,15,"Chronicles of Narnia: Prince Caspian, The",Star Wars: Episode III - Revenge of the Sith,3.2998


In [70]:
print("Group 3 | Average — TOP 3")
df_g3_average.head(3)[["New Movie", "Best Rated Movie", "Predicted Score"]]


Group 3 | Average — TOP 3


,New Movie,Best Rated Movie,Predicted Score
Rank,,,
1,HellBoy II: The Golden Army,Star Wars: Episode III - Revenge of the Sith,4.1740
2,"Incredible Hulk, The",Star Wars: Episode III - Revenge of the Sith,4.0415
3,Quantum of Solace,"Bourne Ultimatum, The",3.8971


### Strategy: Least Misery

In [71]:
df_g3_least_misery = predict_all([11, 15, 29], "least_misery")
print("Group 3 | Least Misery — full ranking:")
df_g3_least_misery


Group 3 | Least Misery — full ranking:


,ID,New Movie,Best Rated Movie,Predicted Score
Rank,,,,
1,9,HellBoy II: The Golden Army,Star Wars: Episode III - Revenge of the Sith,3.5777
2,1,"Incredible Hulk, The",Star Wars: Episode III - Revenge of the Sith,3.4641
3,6,Quantum of Solace,"Bourne Ultimatum, The",3.4641
4,5,Burn After Reading,Pulp Fiction,3.2439
5,2,Indiana Jones and the Kingdom of the Crys...,Pirates of the Caribbean: The Curse of th...,3.0311
6,4,Forgetting Sarah Marshall,There's Something About Mary,3.0000
7,11,Sex and the City,There's Something About Mary,3.0000
8,3,Kung Fu Panda,"Incredibles, The",2.9464
9,15,"Chronicles of Narnia: Prince Caspian, The",Star Wars: Episode III - Revenge of the Sith,2.8284


In [72]:
print("Group 3 | Least Misery — TOP 3")
df_g3_least_misery.head(3)[["New Movie", "Best Rated Movie", "Predicted Score"]]


Group 3 | Least Misery — TOP 3


,New Movie,Best Rated Movie,Predicted Score
Rank,,,
1,HellBoy II: The Golden Army,Star Wars: Episode III - Revenge of the Sith,3.5777
2,"Incredible Hulk, The",Star Wars: Episode III - Revenge of the Sith,3.4641
3,Quantum of Solace,"Bourne Ultimatum, The",3.4641


### Strategy: Most Pleasure

In [73]:
df_g3_most_pleasure = predict_all([11, 15, 29], "most_pleasure")
print("Group 3 | Most Pleasure — full ranking:")
df_g3_most_pleasure


Group 3 | Most Pleasure — full ranking:


,ID,New Movie,Best Rated Movie,Predicted Score
Rank,,,,
1,4,Forgetting Sarah Marshall,There's Something About Mary,4.5000
2,11,Sex and the City,There's Something About Mary,4.5000
3,9,HellBoy II: The Golden Army,Pirates of the Caribbean: The Curse of th...,4.4721
4,1,"Incredible Hulk, The",Star Wars: Episode III - Revenge of the Sith,4.3301
5,2,Indiana Jones and the Kingdom of the Crys...,Pirates of the Caribbean: The Curse of th...,4.3301
6,6,Quantum of Solace,"Bourne Ultimatum, The",4.3301
7,15,"Chronicles of Narnia: Prince Caspian, The",Lord of the Rings: The Return of the King...,4.0825
8,3,Kung Fu Panda,"Incredibles, The",3.8971
9,10,Tropic Thunder,Pirates of the Caribbean: The Curse of th...,3.7500


In [74]:
print("Group 3 | Most Pleasure — TOP 3")
df_g3_most_pleasure.head(3)[["New Movie", "Best Rated Movie", "Predicted Score"]]


Group 3 | Most Pleasure — TOP 3


,New Movie,Best Rated Movie,Predicted Score
Rank,,,
1,Forgetting Sarah Marshall,There's Something About Mary,4.5000
2,Sex and the City,There's Something About Mary,4.5000
3,HellBoy II: The Golden Army,Pirates of the Caribbean: The Curse of th...,4.4721


## 6 · Results — Group 7 (U1, U6, U29)

### Strategy: Average

In [75]:
df_g7_average = predict_all([1, 6, 29], "average")
print("Group 7 | Average — full ranking:")
df_g7_average


Group 7 | Average — full ranking:


,ID,New Movie,Best Rated Movie,Predicted Score
Rank,,,,
1,4,Forgetting Sarah Marshall,Pretty Woman,3.6667
2,11,Sex and the City,Pretty Woman,3.6667
3,5,Burn After Reading,Pulp Fiction,3.3947
4,6,Quantum of Solace,"Bourne Ultimatum, The",3.1754
5,1,"Incredible Hulk, The","Matrix, The",3.0000
6,8,"10,000 B.C.","Dark Knight, The",3.0000
7,9,HellBoy II: The Golden Army,Pirates of the Caribbean: The Curse of th...,2.9814
8,2,Indiana Jones and the Kingdom of the Crys...,Pirates of the Caribbean: The Curse of th...,2.8868
9,15,"Chronicles of Narnia: Prince Caspian, The",Lord of the Rings: The Return of the King...,2.7217


In [76]:
print("Group 7 | Average — TOP 3")
df_g7_average.head(3)[["New Movie", "Best Rated Movie", "Predicted Score"]]


Group 7 | Average — TOP 3


,New Movie,Best Rated Movie,Predicted Score
Rank,,,
1,Forgetting Sarah Marshall,Pretty Woman,3.6667
2,Sex and the City,Pretty Woman,3.6667
3,Burn After Reading,Pulp Fiction,3.3947


### Strategy: Least Misery

In [77]:
df_g7_least_misery = predict_all([1, 6, 29], "least_misery")
print("Group 7 | Least Misery — full ranking:")
df_g7_least_misery


Group 7 | Least Misery — full ranking:


,ID,New Movie,Best Rated Movie,Predicted Score
Rank,,,,
1,5,Burn After Reading,Pulp Fiction,3.2439
2,4,Forgetting Sarah Marshall,There's Something About Mary,3.0000
3,11,Sex and the City,There's Something About Mary,3.0000
4,1,"Incredible Hulk, The",Terminator 2: Judgment Day,2.6922
5,6,Quantum of Solace,"Matrix, The",2.6667
6,8,"10,000 B.C.","Dark Knight, The",2.6667
7,3,Kung Fu Panda,"Monsters, Inc.",2.3333
8,9,HellBoy II: The Golden Army,X-Men,2.3238
9,12,Jumper,"Matrix, The",2.3094


In [78]:
print("Group 7 | Least Misery — TOP 3")
df_g7_least_misery.head(3)[["New Movie", "Best Rated Movie", "Predicted Score"]]


Group 7 | Least Misery — TOP 3


,New Movie,Best Rated Movie,Predicted Score
Rank,,,
1,Burn After Reading,Pulp Fiction,3.2439
2,Forgetting Sarah Marshall,There's Something About Mary,3.0000
3,Sex and the City,There's Something About Mary,3.0000


### Strategy: Most Pleasure

In [79]:
df_g7_most_pleasure = predict_all([1, 6, 29], "most_pleasure")
print("Group 7 | Most Pleasure — full ranking:")
df_g7_most_pleasure


Group 7 | Most Pleasure — full ranking:


,ID,New Movie,Best Rated Movie,Predicted Score
Rank,,,,
1,4,Forgetting Sarah Marshall,Pretty Woman,4.5000
2,11,Sex and the City,Pretty Woman,4.5000
3,6,Quantum of Solace,"Bourne Ultimatum, The",3.8971
4,5,Burn After Reading,Pulp Fiction,3.6742
5,9,HellBoy II: The Golden Army,Pirates of the Caribbean: The Curse of th...,3.5777
6,1,"Incredible Hulk, The",Star Wars: Episode III - Revenge of the Sith,3.4641
7,2,Indiana Jones and the Kingdom of the Crys...,Pirates of the Caribbean: The Curse of th...,3.4641
8,8,"10,000 B.C.","Dark Knight, The",3.3333
9,15,"Chronicles of Narnia: Prince Caspian, The",Lord of the Rings: The Return of the King...,3.2660


In [80]:
print("Group 7 | Most Pleasure — TOP 3")
df_g7_most_pleasure.head(3)[["New Movie", "Best Rated Movie", "Predicted Score"]]


Group 7 | Most Pleasure — TOP 3


,New Movie,Best Rated Movie,Predicted Score
Rank,,,
1,Forgetting Sarah Marshall,Pretty Woman,4.5000
2,Sex and the City,Pretty Woman,4.5000
3,Quantum of Solace,"Bourne Ultimatum, The",3.8971


## 7 · Summary — Top-3 Recommendations

In [81]:
GROUPS = {3: [11, 15, 29], 7: [1, 6, 29]}
STRATS = [("Average","average"),
          ("Least Misery","least_misery"),
          ("Most Pleasure","most_pleasure")]

rows = []
for gid, users in GROUPS.items():
    for label, key in STRATS:
        top3 = predict_all(users, key)
        rows.append({
            "Group"    : f"Group {gid}",
            "Strategy" : label,
            "1st"      : top3.iloc[0]["New Movie"],
            "2nd"      : top3.iloc[1]["New Movie"],
            "3rd"      : top3.iloc[2]["New Movie"],
        })

df_summary = pd.DataFrame(rows).set_index(["Group", "Strategy"])
print("Top-3 Recommendations — all groups and strategies:")
df_summary


Top-3 Recommendations — all groups and strategies:


1st                        2nd  \
Group   Strategy                                                                
Group 3 Average        HellBoy II: The Golden Army       Incredible Hulk, The   
        Least Misery   HellBoy II: The Golden Army       Incredible Hulk, The   
        Most Pleasure    Forgetting Sarah Marshall           Sex and the City   
Group 7 Average          Forgetting Sarah Marshall           Sex and the City   
        Least Misery            Burn After Reading  Forgetting Sarah Marshall   
        Most Pleasure    Forgetting Sarah Marshall           Sex and the City   

                                               3rd  
Group   Strategy                                    
Group 3 Average                  Quantum of Solace  
        Least Misery             Quantum of Solace  
        Most Pleasure  HellBoy II: The Golden Army  
Group 7 Average                 Burn After Reading  
        Least Misery              Sex and the City  
        Most Pleasure            Quantum of Solace